In [2]:
%pip install cvxpy

  Using cached clarabel-0.11.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 3.3 MB/s  0:00:00
Using cached clarabel-0.11.1-cp39-abi3-win_amd64.whl (887 kB)
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.5 MB 4.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.1/7.5 MB 5.1 MB/s eta 0:00:02
   --------------- ------------------------ 2.9/7.5 MB 4.9 MB/s eta 0:00:01
   ------------------- -------------------- 3.7/7.5 MB 4.4 MB/s eta 0:00:01
   --------------------- ------------------ 3.9/7.5 MB 4.3 MB/s eta 0:00:01
   ----------------------- ---------------- 4.5/7.5 MB 3.6 MB/s eta 0:00:01
   ------------------------- -------------- 4.7/7.5 MB 3.4 

In [5]:
import pandas as pd
import numpy as np
import cvxpy as cp
import os
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# MÓDULO 1: O MOTOR MATEMÁTICO DO BLACK-LITTERMAN
# ==============================================================================
def black_litterman_posterior(sigma, w_mkt, Q, omega, P, delta=2.5, tau=0.05):
    """
    Calcula os Retornos Esperados (Mu) e a Covariância Posterior usando a Otimização Reversa.
    """
    # 1. Retorno de Equilíbrio Implícito do Mercado (Pi)
    pi = delta * sigma.dot(w_mkt)
    
    # 2. Inversão de Matrizes (Usamos pseudo-inversa para estabilidade numérica)
    tau_sigma_inv = np.linalg.pinv(tau * sigma)
    omega_inv = np.linalg.pinv(omega)
    
    # 3. Teorema de Bayes (A Magia do Black-Litterman)
    # Termo Esquerdo: [ (tau*Sigma)^-1 + P^T * Omega^-1 * P ]^-1
    termo_esquerdo = np.linalg.pinv(tau_sigma_inv + P.T.dot(omega_inv).dot(P))
    
    # Termo Direito: [ (tau*Sigma)^-1 * Pi + P^T * Omega^-1 * Q ]
    termo_direito = tau_sigma_inv.dot(pi) + P.T.dot(omega_inv).dot(Q)
    
    # Retorno Esperado Posterior (Mu_BL)
    mu_bl = termo_esquerdo.dot(termo_direito)
    
    # Covariância Posterior (Sigma_BL)
    sigma_bl = sigma + termo_esquerdo
    
    return mu_bl, sigma_bl

# ==============================================================================
# MÓDULO 2: OTIMIZADOR CONVEXO COM L1 E LIMITES DE EXPOSIÇÃO
# ==============================================================================
def otimizar_carteira_bl(mu_bl, sigma_bl, pesos_atuais, custo_transacional=0.002, aversao_risco=2.5, limite_por_ativo=0.10):
    n_ativos = len(mu_bl)
    w = cp.Variable(n_ativos)
    
    # Risco e Retorno com base nas previsões do Black-Litterman
    risco = cp.quad_form(w, cp.psd_wrap(sigma_bl + np.eye(n_ativos)*1e-9))
    retorno = mu_bl.T @ w
    
    # Penalidade L1 (Turnover)
    giro_l1 = cp.norm(w - pesos_atuais, 1)
    custo_total = custo_transacional * giro_l1
    
    # Maximizar o Sharpe Posterior descontando custos
    funcao_objetivo = cp.Maximize(retorno - (aversao_risco * risco) - custo_total)
    
    # RESTRITORES INSTITUCIONAIS
    # 1. Soma 100% (Fully Invested)
    # 2. w >= 0 (Sem operar vendido / Long Only)
    # 3. w <= limite_por_ativo (Nenhuma ação pode ter mais de 10% do fundo - CVM)
    restricoes = [cp.sum(w) == 1, w >= 0, w <= limite_por_ativo] 
    
    problema = cp.Problem(funcao_objetivo, restricoes)
    
    try:
        problema.solve(solver=cp.CLARABEL) 
        if w.value is None:
            return pesos_atuais
        
        # Limpeza matemática de resíduos
        pesos_limpos = np.where(w.value < 1e-4, 0, w.value)
        return pesos_limpos / np.sum(pesos_limpos)
    except:
        return pesos_atuais

# ==============================================================================
# MÓDULO 3: BACKTEST WALK-FORWARD (A MÁQUINA DO TEMPO)
# ==============================================================================
def executar_backtest_black_litterman_lstm(diretorio_dados):
    print("=== INÍCIO DO BACKTEST HÍBRIDO: BLACK-LITTERMAN + LSTM ===")
    
    # 1. Carregar Dados
    print("1. A carregar bases de dados (Retornos, Market Cap, Visões LSTM)...")
    df_retornos = pd.read_csv(os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv"), index_col='Data', parse_dates=True)
    #df_pesos_mkt = pd.read_csv(os.path.join(diretorio_dados, "pesos_mercado_black_litterman.csv"), index_col=0)

    # Carregue o arquivo
    df_visoes = pd.read_csv(os.path.join(diretorio_dados,"visoes_lstm_ff5_overnight.csv"))
    # Renomeia a coluna correta e converte para datetime
    if 'Unnamed: 0' in df_visoes.columns:
        df_visoes.rename(columns={'Unnamed: 0': 'Data'}, inplace=True)
    
    df_visoes['Data'] = pd.to_datetime(df_visoes['Data'])
    # ---------------------

    df_pesos_mkt = pd.read_csv(os.path.join(diretorio_dados, "pesos_mercado_black_litterman.csv"), sep=';', decimal=',', index_col=0)


    colunas_ativos = [col for col in df_retornos.columns if col != 'IBOV']
    df_ativos = df_retornos[colunas_ativos]
    n_ativos = len(colunas_ativos)
    
    # 2. Alinhar Vetor de Mercado (w_mkt)
    w_mkt = df_pesos_mkt.reindex(colunas_ativos)['Peso_Mercado_w_mkt'].fillna(0).values
    w_mkt = w_mkt / np.sum(w_mkt) # Garantir 100%
    
    # 3. Preparar Estruturas de Dados
    datas_rebal = np.sort(df_visoes['Data'].unique())
    historico_pesos = pd.DataFrame(index=datas_rebal, columns=colunas_ativos, dtype=float)
    historico_turnover = pd.Series(index=datas_rebal, dtype=float)
    
    pesos_atuais = w_mkt.copy() # O fundo nasce igual ao IBOVESPA
    
    print(f"2. A iniciar a Máquina do Tempo ({len(datas_rebal)} meses)...")
    
    for data_corte in tqdm(datas_rebal):
        # A. Histórico para a Matriz de Covariância (Apenas o passado!)
        dados_janela = df_ativos.loc[:data_corte]
        
        # Sigma Mensalizado (21 dias úteis)
        sigma_mensal = dados_janela.cov().values * 21 
        
        # B. Extrair Visões da Inteligência Artificial para este mês específico
        visoes_mes = df_visoes[df_visoes['Data'] == data_corte].set_index('Ativo').reindex(colunas_ativos)
        
        # Vetor Q (O que a rede acha que vai render)
        Q = visoes_mes['Retorno_Q_Previsto'].fillna(0).values 
        
        # Matriz Omega (Incerteza da rede) - Transformamos num vetor diagonal
        # Adicionamos 1e-6 para garantir que não há divisão por zero (Incerteza não pode ser absoluta)
        erros_mse = visoes_mes['Incerteza_Omega'].fillna(0.1).values + 1e-6
        omega = np.diag(erros_mse)
        
        # Matriz P (Como temos visões para todos os ativos, é uma Matriz Identidade)
        P = np.eye(n_ativos)
        
        # C. Matemática de Black-Litterman
        mu_bl, sigma_bl = black_litterman_posterior(
            sigma=sigma_mensal, 
            w_mkt=w_mkt, 
            Q=Q, 
            omega=omega, 
            P=P, 
            delta=2.5, 
            tau=0.05
        )
        
        # D. Otimização Final do Mês
        novos_pesos = otimizar_carteira_bl(
            mu_bl=mu_bl, 
            sigma_bl=sigma_bl, 
            pesos_atuais=pesos_atuais, 
            custo_transacional=0.002 # 0.2% de corretagem realista
        )
        
        turnover = np.sum(np.abs(novos_pesos - pesos_atuais)) / 2.0
        
        # E. Registar Resultados
        historico_pesos.loc[data_corte] = novos_pesos
        historico_turnover.loc[data_corte] = turnover
        pesos_atuais = novos_pesos.copy()

    # 4. Salvar tudo
    print("\n3. A exportar o portfólio definitivo...")
    caminho_hist = os.path.join(diretorio_dados, "historico_alocacao_lstm_ff5_overnight.csv")
    caminho_turn = os.path.join(diretorio_dados, "historico_turnover_lstm_ff5_overnight.csv")
    
    historico_pesos.to_csv(caminho_hist)
    historico_turnover.to_csv(caminho_turn)
    
    print("=== SÍNTESE DO MODELO FINAL ===")
    print(f"Turnover médio mensal: {historico_turnover.mean()*100:.2f}%")
    print(f"Pesos salvos em: {caminho_hist}")
    
    return historico_pesos

# ==========================================
# ÁREA DE EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
pesos_bl_lstm = executar_backtest_black_litterman_lstm(pasta_dados)

=== INÍCIO DO BACKTEST HÍBRIDO: BLACK-LITTERMAN + LSTM ===
1. A carregar bases de dados (Retornos, Market Cap, Visões LSTM)...
2. A iniciar a Máquina do Tempo (132 meses)...


100%|██████████| 132/132 [00:10<00:00, 12.12it/s]


3. A exportar o portfólio definitivo...
=== SÍNTESE DO MODELO FINAL ===
Turnover médio mensal: 9.45%
Pesos salvos em: C:\VSCodeWorkspace\TCC_Escrito\data\historico_alocacao_lstm_ff5_overnight.csv
